# **Imports & Installs**

In [ ]:
!pip install transformers
!pip install accelerate -U

import logging
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import Trainer, TrainingArguments
from transformers import TextDataset, DataCollatorForLanguageModeling

# **Finetuning Model**

In [ ]:
def fine_tune_gpt2_querygeneration(model_name, train_file, output_dir, eval_file=None):
    logging.basicConfig(level=logging.INFO)

    # Load model and tokenizer
    model = BioGptForCausalLM.from_pretrained(model_name)
    tokenizer = BioGptTokenizer.from_pretrained(model_name)

    # Load datasets
    train_dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=train_file,
        block_size=512)

    eval_dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=eval_file,
        block_size=512) if eval_file else None

    # Create data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        report_to=[],
        overwrite_output_dir=True,
        num_train_epochs=5,
        per_device_train_batch_size=4,
        save_steps=10_000,
        save_total_limit=2,
        evaluation_strategy="epoch" if eval_file else "no",
        logging_dir=f"{output_dir}/logs",
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    trainer.train()

    # Save model and tokenizer
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
